# Trees, Ensembles, and Why XGBoost Works


<a href="https://www.kaggle.com/code/addarm/trees-ensembles-and-why-xgboost-works" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>


![Trees, Ensembles, and XGBoost](https://raw.githubusercontent.com/adamd1985/quant_research/refs/heads/main/images/trees_xgboost_banner.png)


# Introduction

In the [linear regression article](deep_linear_regression.ipynb), we studied models that explain the target with one global equation. In [penalized regressions](penalized_regressions.ipynb), we saw how regularization helps when that equation becomes unstable.

Tree models solve a different problem. Instead of fitting one formula everywhere, they keep asking small questions such as:

- Is `bmi` above some threshold?
- Is `worst radius` large or small?
- Once we split the sample, should we split again?

So a tree is easier to picture than a linear model: it keeps dividing the predictor space into smaller regions, and each final region makes its own prediction.

This notebook builds the topic slowly:

1. understand a single decision tree,
2. see why trees overfit,
3. fix that with pruning,
4. reduce variance with bagging and random forests,
5. reduce bias with boosting,
6. understand how XGBoost makes boosting more disciplined.

The goal is not just to run library calls. It is to understand what each method is trying to fix, and why the next method in the sequence exists at all.

So there are really two readers in mind:

- the **student**, who wants each idea explained in order,
- the **academic critic**, who wants each claim stated carefully enough to be defensible.

This notebook tries to satisfy both. We will go step by step, but we will also be explicit about what a method does, what it does **not** do, and why the stronger methods were invented.


## Bias-Variance Again, but for Trees

Before looking at the formula, keep this picture in mind:

- a tree with too few splits is **too blunt**,
- a tree with too many splits is **too reactive**.

The first problem is underfitting. The second is overfitting.

The bias-variance decomposition is not unique to linear models. Suppose a predictor trained on a random sample is denoted by $\hat f(x)$. At a fixed input $x$, the expected prediction error under squared loss can be written as

$$
\mathbb{E}\big[(Y - \hat f(x))^2 \mid X=x\big]
=
\sigma^2
+
\big(\mathbb{E}[\hat f(x)] - f(x)\big)^2
+
\mathbb{V}\mathrm{ar}(\hat f(x)),
$$

where $\sigma^2$ is irreducible noise, the middle term is squared bias, and the last term is variance.

For the academic reader, one caveat matters: this is the usual **pointwise squared-error decomposition**. We use it here because it gives the cleanest language for talking about tree depth, pruning, and averaging.

For tree models, the most important intuition is this:

- A **small tree** is usually too simple. It misses structure. That means high bias.
- A **very deep tree** can react to tiny quirks of the training set. That means high variance.

So trees are a very visual way to learn bias-variance tradeoffs. Each new split makes the model more flexible, but also more fragile.

The rest of the notebook is really one long answer to a student-friendly question:

**How do we keep the useful flexibility of trees without letting them become too unstable?**

The answer comes in stages:

1. **Pruning** cuts back a tree that has grown too much.
2. **Bagging** averages many unstable trees.
3. **Random forests** make those trees less correlated.
4. **Boosting** adds small corrective trees one by one.
5. **XGBoost** adds explicit regularization and a more careful optimization rule.


# Notebook Setup

We import the standard numerical stack, scikit-learn estimators for the reference implementations, and a few plotting utilities. A fixed seed is used throughout. The notebook is written so that the official `xgboost` package is optional at import time, but the validation section will automatically activate if it is installed.


In [ ]:
import importlib.util
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.calibration import CalibrationDisplay
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.ensemble import (
    BaggingClassifier,
    BaggingRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore")

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])

HAS_XGBOOST = importlib.util.find_spec("xgboost") is not None
if HAS_XGBOOST:
    import xgboost as xgb

SEED = 7
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", __import__("sklearn").__version__)
print("xgboost available:", HAS_XGBOOST)

try:
    from sklearn.metrics import root_mean_squared_error

    def rmse(y_true, y_pred):
        return root_mean_squared_error(y_true, y_pred)
except ImportError:

    def rmse(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)


# The Datasets

To keep the notebook fully reproducible, we use two classical scikit-learn datasets:

1. The **diabetes** dataset for regression, where the target is a quantitative measure of disease progression one year after baseline.
2. The **breast cancer Wisconsin** dataset for binary classification, where the target is malignant versus benign diagnosis.

This split is deliberate. Regression trees predict leaf means and are evaluated with squared-error style metrics, while classification trees predict class probabilities and are evaluated with proper scoring rules such as log loss and Brier score in addition to accuracy and AUC.

That gives us a more honest learning experience: the same algorithmic ideas appear in both settings, but the evaluation language changes with the task.


In [ ]:
X_reg, y_reg = load_diabetes(return_X_y=True, as_frame=True)
X_clf, y_clf = load_breast_cancer(return_X_y=True, as_frame=True)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.25, random_state=SEED)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_clf, y_clf, test_size=0.25, random_state=SEED, stratify=y_clf)

reg_summary = pd.DataFrame(
    {
        "shape": [X_reg.shape],
        "target_mean": [float(np.mean(y_reg))],
        "target_std": [float(np.std(y_reg, ddof=1))],
    },
    index=["Diabetes regression"],
)
clf_summary = pd.DataFrame(
    {
        "shape": [X_clf.shape],
        "positive_rate": [float(np.mean(y_clf))],
        "feature_count": [X_clf.shape[1]],
    },
    index=["Breast cancer classification"],
)

display(reg_summary)
display(clf_summary)
X_reg.head()


# Decision Trees

A decision tree is a sequence of if-then questions. At each node we pick one feature and one threshold, then split the sample into two groups. In symbols, if we split on feature $j$ at threshold $s$, we create

$$
R_1(j, s) = \{x : x_j \le s\}, \qquad
R_2(j, s) = \{x : x_j > s\}.
$$

Then we repeat the same process inside each child node. This is why the method is called **recursive binary splitting**.

The final nodes are called **leaves**. A leaf does not ask another question. It simply returns a prediction.

For regression, the leaf prediction is the mean response in that region:

$$
\hat c_m = \frac{1}{|R_m|}\sum_{i \in R_m} y_i.
$$

So the tree is piecewise constant: every point inside the same leaf gets the same fitted value.

To decide where to split, the tree searches for the question that makes the two children as pure as possible. In regression this means minimizing residual sum of squares:

$$
\min_{j,s}
\left[
\sum_{i:x_i \in R_1(j,s)} (y_i - \hat c_1)^2
+
\sum_{i:x_i \in R_2(j,s)} (y_i - \hat c_2)^2
\right].
$$

For classification, we instead minimize an impurity criterion such as Gini impurity,

$$
G(t) = \sum_{k=1}^{K} \hat p_{tk} (1 - \hat p_{tk}),
$$

or entropy,

$$
H(t) = -\sum_{k=1}^{K} \hat p_{tk} \log \hat p_{tk}.
$$

These criteria reward splits that make the child nodes more homogeneous than the parent.

A good mental model is:

- regression tree: "split so the responses inside each side look similar",
- classification tree: "split so the classes inside each side are less mixed".

The key academic point is that the tree is greedy: it chooses the best split *right now*, not the globally best tree all at once. That makes trees fast and practical, but also explains why they can be unstable.


In [ ]:
tree_reg_demo = DecisionTreeRegressor(max_depth=2, random_state=SEED)
tree_clf_demo = DecisionTreeClassifier(max_depth=2, random_state=SEED)

tree_reg_demo.fit(Xr_train, yr_train)
tree_clf_demo.fit(Xc_train, yc_train)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plot_tree(
    tree_reg_demo,
    feature_names=X_reg.columns,
    filled=True,
    rounded=True,
    ax=axes[0],
    fontsize=8,
)
axes[0].set_title("Regression tree with max_depth=2")

plot_tree(
    tree_clf_demo,
    feature_names=X_clf.columns,
    class_names=["malignant", "benign"],
    filled=True,
    rounded=True,
    ax=axes[1],
    fontsize=7,
)
axes[1].set_title("Classification tree with max_depth=2")
plt.tight_layout()

print("Regression leaf count:", tree_reg_demo.get_n_leaves())
print("Classification leaf count:", tree_clf_demo.get_n_leaves())


## Why Trees Overfit

A deep tree has a lot of freedom. That is useful when the signal is genuinely nonlinear, but dangerous when the tree starts fitting random accidents in the training set.

So we should expect the following pattern:

- training error keeps improving as depth grows,
- test error improves only up to a point,
- after that point, extra depth mostly memorizes noise.

The next cell shows that pattern for both regression and classification.


In [ ]:
depths = range(1, 13)
reg_rows = []
clf_rows = []

for depth in depths:
    reg_model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    reg_model.fit(Xr_train, yr_train)
    reg_rows.append(
        {
            "depth": depth,
            "train_rmse": rmse(yr_train, reg_model.predict(Xr_train)),
            "test_rmse": rmse(yr_test, reg_model.predict(Xr_test)),
        }
    )

    clf_model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    clf_model.fit(Xc_train, yc_train)
    clf_rows.append(
        {
            "depth": depth,
            "train_logloss": log_loss(yc_train, clf_model.predict_proba(Xc_train)[:, 1]),
            "test_logloss": log_loss(yc_test, clf_model.predict_proba(Xc_test)[:, 1]),
        }
    )

reg_depth_df = pd.DataFrame(reg_rows)
clf_depth_df = pd.DataFrame(clf_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(reg_depth_df["depth"], reg_depth_df["train_rmse"], marker="o", label="Train RMSE")
axes[0].plot(reg_depth_df["depth"], reg_depth_df["test_rmse"], marker="o", label="Test RMSE")
axes[0].set_title("Regression tree depth path")
axes[0].set_xlabel("Max depth")
axes[0].set_ylabel("RMSE")
axes[0].legend()

axes[1].plot(clf_depth_df["depth"], clf_depth_df["train_logloss"], marker="o", label="Train log loss")
axes[1].plot(clf_depth_df["depth"], clf_depth_df["test_logloss"], marker="o", label="Test log loss")
axes[1].set_title("Classification tree depth path")
axes[1].set_xlabel("Max depth")
axes[1].set_ylabel("Log loss")
axes[1].legend()
plt.tight_layout()

display(reg_depth_df)
display(clf_depth_df)


A student-friendly way to read those plots is:

- if the training curve improves and the test curve also improves, the model is learning useful structure,
- if the training curve improves but the test curve gets worse, the extra flexibility is mostly fitting noise.

That widening gap is the visual signature of overfitting.


## Cost-Complexity Pruning

If a tree is too large, one option is not to grow it that far in the first place. Another option, which is conceptually cleaner for teaching, is:

1. grow a large tree,
2. then cut it back.

Breiman et al. formalized that idea with the cost-complexity objective

$$
C_\alpha(T) = \sum_{m=1}^{|T|} \sum_{i \in R_m} L(y_i, \hat c_m) + \alpha |T|,
$$

where $|T|$ is the number of leaves and $\alpha \ge 0$ penalizes complexity.

Read this objective in plain English:

- the first term rewards fit,
- the second term punishes too many leaves.

Large `ccp_alpha` means "be more skeptical about adding structure." Small `ccp_alpha` means "allow a more detailed tree."


In [ ]:
reg_path_model = DecisionTreeRegressor(random_state=SEED)
reg_path_model.fit(Xr_train, yr_train)
reg_path = reg_path_model.cost_complexity_pruning_path(Xr_train, yr_train)

clf_path_model = DecisionTreeClassifier(random_state=SEED)
clf_path_model.fit(Xc_train, yc_train)
clf_path = clf_path_model.cost_complexity_pruning_path(Xc_train, yc_train)

reg_alphas = np.unique(np.quantile(reg_path.ccp_alphas, np.linspace(0, 0.95, 12)))
clf_alphas = np.unique(np.quantile(clf_path.ccp_alphas, np.linspace(0, 0.95, 12)))

reg_cv = []
for alpha in reg_alphas:
    model = DecisionTreeRegressor(random_state=SEED, ccp_alpha=float(alpha))
    score = -cross_val_score(
        model,
        Xr_train,
        yr_train,
        cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring="neg_root_mean_squared_error",
    ).mean()
    reg_cv.append({"ccp_alpha": float(alpha), "cv_rmse": float(score)})

clf_cv = []
for alpha in clf_alphas:
    model = DecisionTreeClassifier(random_state=SEED, ccp_alpha=float(alpha))
    score = -cross_val_score(
        model,
        Xc_train,
        yc_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring="neg_log_loss",
    ).mean()
    clf_cv.append({"ccp_alpha": float(alpha), "cv_logloss": float(score)})

reg_prune_df = pd.DataFrame(reg_cv)
clf_prune_df = pd.DataFrame(clf_cv)

best_reg_alpha = float(reg_prune_df.loc[reg_prune_df["cv_rmse"].idxmin(), "ccp_alpha"])
best_clf_alpha = float(clf_prune_df.loc[clf_prune_df["cv_logloss"].idxmin(), "ccp_alpha"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(reg_prune_df["ccp_alpha"], reg_prune_df["cv_rmse"], marker="o")
axes[0].axvline(best_reg_alpha, color="black", linestyle="--", label=f"best={best_reg_alpha:.4f}")
axes[0].set_title("Regression pruning path")
axes[0].set_xlabel("ccp_alpha")
axes[0].set_ylabel("CV RMSE")
axes[0].legend()

axes[1].plot(clf_prune_df["ccp_alpha"], clf_prune_df["cv_logloss"], marker="o")
axes[1].axvline(best_clf_alpha, color="black", linestyle="--", label=f"best={best_clf_alpha:.4f}")
axes[1].set_title("Classification pruning path")
axes[1].set_xlabel("ccp_alpha")
axes[1].set_ylabel("CV log loss")
axes[1].legend()
plt.tight_layout()

print("Best regression ccp_alpha:", best_reg_alpha)
print("Best classification ccp_alpha:", best_clf_alpha)


The pruning plot gives us a practical rule:

- far left: almost no penalty, tree can stay too large,
- far right: too much penalty, tree becomes too simple,
- middle region near the best cross-validated score: best compromise.

So `ccp_alpha` is not a mysterious knob. It is just the price we charge for extra tree complexity.


# Bagging

Bagging, short for bootstrap aggregating, is the first big ensemble idea in this notebook.

Start from one fact: a single deep tree is unstable. If we perturb the sample, the fitted tree can change a lot.

Bagging answers: what if we fit **many** trees on slightly different versions of the data, then average them?

Formally, if we draw bootstrap samples and fit one tree on each sample, the regression predictor is

$$
\hat f_{\text{bag}}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat f^{*(b)}(x).
$$

For classification, we average probabilities or use a majority vote.

The key idea is simple:

- each tree is noisy,
- but not noisy in exactly the same way,
- so averaging reduces variance.

Notice what bagging does **not** do: it does not simplify the individual tree. Each tree can still be large. The stabilization comes from averaging, not from making one tree safer.


In [ ]:
bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=SEED),
    n_estimators=250,
    bootstrap=True,
    random_state=SEED,
    n_jobs=1,
)
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=250,
    bootstrap=True,
    random_state=SEED,
    n_jobs=1,
)

bag_reg.fit(Xr_train, yr_train)
bag_clf.fit(Xc_train, yc_train)

bagging_demo = pd.DataFrame(
    {
        "model": ["Single deep tree", "Bagging"],
        "regression_rmse": [
            rmse(
                yr_test,
                DecisionTreeRegressor(random_state=SEED).fit(Xr_train, yr_train).predict(Xr_test),
            ),
            rmse(yr_test, bag_reg.predict(Xr_test)),
        ],
        "classification_logloss": [
            log_loss(
                yc_test,
                DecisionTreeClassifier(random_state=SEED).fit(Xc_train, yc_train).predict_proba(Xc_test)[:, 1],
            ),
            log_loss(yc_test, bag_clf.predict_proba(Xc_test)[:, 1]),
        ],
    }
)
bagging_demo


How should a student read that output?

If bagging improves test performance relative to one deep tree, the lesson is not "more complexity won." The lesson is "averaging stabilized an unstable learner."

That is the academic reason bagging matters: it changes the variance of the predictor without needing a new splitting rule.


# Random Forests

Bagging helps, but there is still a problem: many trees may keep choosing the same strong predictor near the root. If that happens, the trees are still too similar to each other.

Random forests fix this by adding **feature randomness**. At each split, the tree is not allowed to inspect every predictor. It only sees a random subset. This forces different trees to explore different views of the data.

So the improvement over bagging is:

- bagging: change the rows through bootstrap sampling,
- random forest: change the rows **and** partially change the features each split can inspect.

A useful side product is the **out-of-bag** estimate. Because each tree misses about one third of the sample, those omitted points act like a built-in validation set. We also compare impurity importance with permutation importance, because students should know that feature importance is not a single unquestionable number.


In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=400,
    max_features="sqrt",
    oob_score=True,
    random_state=SEED,
    n_jobs=1,
)
rf_clf = RandomForestClassifier(
    n_estimators=400,
    max_features="sqrt",
    oob_score=True,
    random_state=SEED,
    n_jobs=1,
)

rf_reg.fit(Xr_train, yr_train)
rf_clf.fit(Xc_train, yc_train)

reg_perm = permutation_importance(rf_reg, Xr_test, yr_test, n_repeats=20, random_state=SEED, n_jobs=1)
clf_perm = permutation_importance(rf_clf, Xc_test, yc_test, n_repeats=20, random_state=SEED, n_jobs=1)

reg_importance = (
    pd.DataFrame(
        {
            "impurity": rf_reg.feature_importances_,
            "permutation": reg_perm.importances_mean,
        },
        index=X_reg.columns,
    )
    .sort_values("permutation", ascending=False)
    .head(10)
)

clf_importance = (
    pd.DataFrame(
        {
            "impurity": rf_clf.feature_importances_,
            "permutation": clf_perm.importances_mean,
        },
        index=X_clf.columns,
    )
    .sort_values("permutation", ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
reg_importance.sort_values("permutation").plot.barh(ax=axes[0], color=["#4c78a8", "#f58518"])
axes[0].set_title(f"Regression forest importance | OOB R^2 = {rf_reg.oob_score_:.3f}")
axes[0].set_xlabel("Importance")

clf_importance.sort_values("permutation").plot.barh(ax=axes[1], color=["#54a24b", "#e45756"])
axes[1].set_title(f"Classification forest importance | OOB acc = {rf_clf.oob_score_:.3f}")
axes[1].set_xlabel("Importance")
plt.tight_layout()

display(reg_importance)
display(clf_importance)


Two cautionary lessons belong here.

First, random forests often improve prediction because they make the trees less correlated, not because each tree is individually better.

Second, feature importance should be interpreted carefully. Impurity importance and permutation importance can disagree, especially when predictors overlap in the information they carry. That disagreement is not a bug. It is telling us that "importance" depends on the question being asked.


# Gradient Boosting

Bagging and random forests mainly attack variance. Boosting has a different personality: it attacks bias by building a model in small corrective steps.

Instead of fitting many independent trees and averaging them, boosting fits trees **sequentially**. Each new tree tries to repair what the current model is still getting wrong.

We write the model as

$$
F_M(x) = \sum_{m=0}^{M} \nu h_m(x),
$$

where each $h_m$ is a small tree and $\nu \in (0, 1]$ is the learning rate.

Under squared loss, the idea is especially easy to follow. The negative gradient at stage $m-1$ is just the residual:

$$
r_{im} = y_i - F_{m-1}(x_i).
$$

So regression boosting can be read almost literally:

1. fit a rough model,
2. compute what it got wrong,
3. fit a small tree to those mistakes,
4. add only a fraction of that correction,
5. repeat.

A small learning rate usually makes the updates more conservative and easier to regularize.

Student version: each tree is a correction.

Academic version: the model is built by stagewise additive optimization in function space, and the learning rate controls how aggressively each stage is allowed to move the fit.


In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.85,
    random_state=SEED,
)
gbr.fit(Xr_train, yr_train)

staged_rmse = [rmse(yr_test, pred) for pred in gbr.staged_predict(Xr_test)]

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(staged_rmse) + 1), staged_rmse, color="#4c78a8")
plt.title("Gradient boosting regression: staged test RMSE")
plt.xlabel("Number of trees")
plt.ylabel("Test RMSE")
plt.show()

print("Best staged test RMSE:", float(np.min(staged_rmse)))


The staged error curve is one of the clearest boosting diagnostics.

- Early rounds usually capture the big mistakes.
- Later rounds add finer corrections.
- If the curve starts rising again, the model is beginning to over-correct.

So boosting is not just "add more trees forever." The number of boosting rounds is itself a regularization choice.


## Gradient Boosting for Classification

For classification, the same logic remains, but the notion of "mistake" changes. We are no longer predicting a real-valued target directly. We are predicting a probability.

So instead of plain residuals, we use the gradient of a classification loss, usually logistic loss. Let the current margin be $F_{m-1}(x)$. Then the probability model is

$$
p(x) = \sigma(F_{m-1}(x)) = \frac{1}{1 + e^{-F_{m-1}(x)}}.
$$

The new tree is therefore correcting probability errors, not just class labels. This is why a boosted classifier should be read as a probability model first, and a hard-label predictor second.

For students, the easiest memory aid is:

- regression boosting fixes residuals,
- classification boosting fixes probabilities through the loss gradient.


In [ ]:
gbc = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.85,
    random_state=SEED,
)
gbc.fit(Xc_train, yc_train)

staged_logloss = [log_loss(yc_test, proba[:, 1]) for proba in gbc.staged_predict_proba(Xc_test)]

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(staged_logloss) + 1), staged_logloss, color="#e45756")
plt.title("Gradient boosting classification: staged test log loss")
plt.xlabel("Number of trees")
plt.ylabel("Test log loss")
plt.show()

print("Best staged test log loss:", float(np.min(staged_logloss)))
print(classification_report(yc_test, gbc.predict(Xc_test), target_names=["malignant", "benign"]))


Notice that classification is evaluated with log loss as well as hard-label metrics. That is deliberate.

Accuracy only asks whether the final class is right. Log loss asks whether the model was *confident in the right way*. For boosted classifiers, that probability perspective is essential.


# XGBoost

This is the section where many students get lost, so let us slow it down.

Basic gradient boosting already has a simple story:

1. fit a model,
2. look at what it gets wrong,
3. add a small tree that corrects part of that error.

XGBoost keeps that story, but makes each step more explicit and more disciplined.

A student version is:

- ordinary boosting says: "add the next useful correction,"
- XGBoost says: "add the next useful correction, but charge for complexity and use more information about the loss."

An academic version is:

- ordinary boosting is a stagewise additive procedure,
- XGBoost writes a regularized objective for each new tree and optimizes a second-order approximation to that objective.

At boosting step $t$, we add a tree $f_t$ and minimize

$$
\mathcal{L}^{(t)} =
\sum_{i=1}^{n} l\big(y_i, \hat y_i^{(t-1)} + f_t(x_i)\big)
+
\Omega(f_t).
$$

Read this before reading the symbols:

- the first term says: "fit the data better,"
- the second term says: "do not let the new tree become too wild."

The regularizer is

$$
\Omega(f) = \gamma T + \frac{\lambda}{2} \sum_{j=1}^{T} w_j^2.
$$

Here:

- $T$ is the number of leaves,
- $w_j$ is the prediction attached to leaf $j$,
- $\gamma$ charges for adding leaves,
- $\lambda$ shrinks leaf values toward zero.

So even before we look at the split formula, we already know what XGBoost is trying to do: improve fit, but make each extra piece of structure pay a price.

The next idea is the one that makes XGBoost feel more technical than ordinary boosting. Instead of only looking at first-order errors, it uses a second-order Taylor approximation of the loss:

$$
\mathcal{L}^{(t)}
\approx
\sum_{i=1}^{n}
\left[
g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2
\right]
+
\Omega(f_t),
$$

where

$$
g_i = \partial_{\hat y^{(t-1)}} l(y_i, \hat y_i^{(t-1)}), \qquad
h_i = \partial^2_{\hat y^{(t-1)}} l(y_i, \hat y_i^{(t-1)}).
$$

Student reading:

- $g_i$ says which direction the prediction should move,
- $h_i$ says how sensitive the loss is around that point.

Academic reading:

- the update is based on a local second-order approximation,
- so split selection and leaf weights are tied directly to the approximated objective, not to a heuristic impurity score.

Once we commit to a leaf, XGBoost can solve for its best value in closed form:

$$
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}.
$$

This formula is worth pausing on. It says:

- if the gradients in a leaf strongly point in one direction, the leaf should move in that direction,
- but large Hessian mass and large $\lambda$ keep that move under control.

Now comes the split-gain formula:

$$
\mathrm{Gain}
=
\frac{1}{2}
\left(
\frac{G_L^2}{H_L + \lambda}
+
\frac{G_R^2}{H_R + \lambda}
-
\frac{(G_L + G_R)^2}{H_L + H_R + \lambda}
\right)
- \gamma.
$$

Do not try to memorize it mechanically. Read it as a decision rule:

- compare the score of keeping everything in one node,
- against the score of splitting into left and right children,
- then subtract the complexity price $\gamma$.

So the formula answers one practical question only:

**Is this split worth paying for?**

What should a student remember after this section?

- XGBoost is still boosting,
- but each new tree is chosen with a regularized objective,
- leaf values are shrunk analytically,
- and splits are accepted only if they improve the objective enough to justify their complexity.

What should the academic critic remember?

- the method replaces heuristic split scoring with objective-based scoring,
- uses second-order information,
- and makes regularization part of the optimization itself, not an afterthought.


## Manual XGBoost Core

The implementation below is intentionally a *teaching implementation*.

For the student, that means: this code is here to make the ideas visible.

For the critic, that means: this code is not claiming to be a production reimplementation of the library.

What it **does** include:

1. first- and second-order gradients,
2. regularized leaf weights,
3. gain-based split selection,
4. shrinkage through a learning rate,
5. row and column subsampling,
6. early stopping on an internal validation split.

What it **does not** include:

- histogram acceleration,
- sparsity-aware default directions,
- weighted quantile sketching,
- cache-aware data structures,
- out-of-core execution,
- distributed training.

So the purpose of the manual model is narrow but important:

- understand the statistical engine,
- then compare it to the official implementation,
- without pretending we have rebuilt the full industrial system.


In [ ]:
@dataclass
class TreeNode:
    feature: int | None = None
    threshold: float | None = None
    left: "TreeNode | None" = None
    right: "TreeNode | None" = None
    value: float | None = None


def _sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))


class SimpleXGBTree:
    def __init__(
        self,
        max_depth=2,
        min_samples_leaf=10,
        reg_lambda=1.0,
        gamma=0.0,
        max_bins=32,
        feature_indices=None,
    ):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.max_bins = max_bins
        self.feature_indices = feature_indices
        self.root_ = None

    def _leaf_value(self, grad, hess):
        return -grad.sum() / (hess.sum() + self.reg_lambda)

    def _score(self, grad, hess):
        return (grad.sum() ** 2) / (hess.sum() + self.reg_lambda)

    def _candidate_thresholds(self, values):
        uniq = np.unique(values)
        if len(uniq) <= 1:
            return np.array([])
        thresholds = (uniq[:-1] + uniq[1:]) / 2.0
        if len(thresholds) > self.max_bins:
            idx = np.linspace(0, len(thresholds) - 1, self.max_bins, dtype=int)
            thresholds = thresholds[idx]
        return thresholds

    def _best_split(self, X, grad, hess):
        best = None
        parent_score = self._score(grad, hess)
        n_features = X.shape[1]

        for local_j in range(n_features):
            values = X[:, local_j]
            for threshold in self._candidate_thresholds(values):
                left_mask = values <= threshold
                right_mask = ~left_mask
                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue

                gain = (
                    0.5 * (self._score(grad[left_mask], hess[left_mask]) + self._score(grad[right_mask], hess[right_mask]) - parent_score)
                    - self.gamma
                )

                if best is None or gain > best["gain"]:
                    best = {
                        "gain": float(gain),
                        "feature": int(self.feature_indices[local_j]),
                        "threshold": float(threshold),
                        "left_mask": left_mask,
                        "right_mask": right_mask,
                    }
        return best

    def _build(self, X, grad, hess, depth):
        if depth >= self.max_depth or X.shape[0] < 2 * self.min_samples_leaf:
            return TreeNode(value=float(self._leaf_value(grad, hess)))

        split = self._best_split(X, grad, hess)
        if split is None or split["gain"] <= 0:
            return TreeNode(value=float(self._leaf_value(grad, hess)))

        left = self._build(
            X[split["left_mask"]],
            grad[split["left_mask"]],
            hess[split["left_mask"]],
            depth + 1,
        )
        right = self._build(
            X[split["right_mask"]],
            grad[split["right_mask"]],
            hess[split["right_mask"]],
            depth + 1,
        )
        return TreeNode(
            feature=split["feature"],
            threshold=split["threshold"],
            left=left,
            right=right,
        )

    def fit(self, X, grad, hess):
        self.root_ = self._build(X, grad, hess, depth=0)
        return self

    def _predict_row(self, row, node):
        if node.value is not None:
            return node.value
        if row[node.feature] <= node.threshold:
            return self._predict_row(row, node.left)
        return self._predict_row(row, node.right)

    def predict(self, X):
        X = np.asarray(X)
        return np.array([self._predict_row(row, self.root_) for row in X])


class _ManualXGBBase(BaseEstimator):
    def __init__(
        self,
        n_estimators=60,
        learning_rate=0.08,
        max_depth=2,
        min_samples_leaf=10,
        subsample=0.8,
        colsample=0.8,
        reg_lambda=1.0,
        gamma=0.0,
        max_bins=24,
        validation_fraction=0.2,
        early_stopping_rounds=10,
        random_state=7,
    ):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.subsample = subsample
        self.colsample = colsample
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.max_bins = max_bins
        self.validation_fraction = validation_fraction
        self.early_stopping_rounds = early_stopping_rounds
        self.random_state = random_state

    def _fit_booster(self, X, y, objective):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        rng = np.random.default_rng(self.random_state)

        n = X.shape[0]
        idx = np.arange(n)
        rng.shuffle(idx)
        split = max(int((1 - self.validation_fraction) * n), 1)
        train_idx = idx[:split]
        val_idx = idx[split:] if split < n else idx[:0]

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        if objective == "regression":
            self.base_score_ = float(np.mean(y_train))
            train_margin = np.full(X_train.shape[0], self.base_score_)
            val_margin = np.full(X_val.shape[0], self.base_score_) if len(val_idx) else np.array([])
        else:
            p0 = np.clip(np.mean(y_train), 1e-5, 1 - 1e-5)
            self.base_score_ = float(np.log(p0 / (1 - p0)))
            train_margin = np.full(X_train.shape[0], self.base_score_)
            val_margin = np.full(X_val.shape[0], self.base_score_) if len(val_idx) else np.array([])

        self.trees_ = []
        self.train_scores_ = []
        self.val_scores_ = []
        best_val = np.inf
        best_iter = -1
        patience = 0

        for _ in range(self.n_estimators):
            if objective == "regression":
                grad = train_margin - y_train
                hess = np.ones_like(grad)
            else:
                p = _sigmoid(train_margin)
                grad = p - y_train
                hess = np.maximum(p * (1 - p), 1e-6)

            row_count = max(int(self.subsample * X_train.shape[0]), 2 * self.min_samples_leaf)
            row_idx = rng.choice(X_train.shape[0], size=min(row_count, X_train.shape[0]), replace=False)

            feat_count = max(int(self.colsample * X_train.shape[1]), 1)
            feat_idx = np.sort(rng.choice(X_train.shape[1], size=feat_count, replace=False))

            tree = SimpleXGBTree(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                reg_lambda=self.reg_lambda,
                gamma=self.gamma,
                max_bins=self.max_bins,
                feature_indices=feat_idx,
            )
            tree.fit(X_train[row_idx][:, feat_idx], grad[row_idx], hess[row_idx])

            train_margin = train_margin + self.learning_rate * tree.predict(X_train)
            self.trees_.append(tree)

            train_loss = (
                mean_squared_error(y_train, train_margin) if objective == "regression" else log_loss(y_train, _sigmoid(train_margin))
            )
            self.train_scores_.append(float(train_loss))

            if len(val_idx):
                val_margin = val_margin + self.learning_rate * tree.predict(X_val)
                val_loss = mean_squared_error(y_val, val_margin) if objective == "regression" else log_loss(y_val, _sigmoid(val_margin))
                self.val_scores_.append(float(val_loss))
                if val_loss + 1e-8 < best_val:
                    best_val = float(val_loss)
                    best_iter = len(self.trees_) - 1
                    patience = 0
                else:
                    patience += 1
                if patience >= self.early_stopping_rounds:
                    break

        if len(val_idx) and best_iter >= 0:
            self.trees_ = self.trees_[: best_iter + 1]
            self.train_scores_ = self.train_scores_[: best_iter + 1]
            self.val_scores_ = self.val_scores_[: best_iter + 1]
            self.best_iteration_ = best_iter + 1
        else:
            self.best_iteration_ = len(self.trees_)
        return self

    def _raw_predict(self, X):
        X = np.asarray(X, dtype=float)
        pred = np.full(X.shape[0], self.base_score_, dtype=float)
        for tree in self.trees_:
            pred += self.learning_rate * tree.predict(X)
        return pred


class ManualXGBRegressor(_ManualXGBBase, RegressorMixin):
    def fit(self, X, y):
        return self._fit_booster(X, y, objective="regression")

    def predict(self, X):
        return self._raw_predict(X)


class ManualXGBClassifier(_ManualXGBBase, ClassifierMixin):
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self._fit_booster(X, y, objective="classification")

    def predict_proba(self, X):
        margin = self._raw_predict(X)
        p = _sigmoid(margin)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [ ]:
manual_xgb_reg = ManualXGBRegressor(
    n_estimators=80,
    learning_rate=0.08,
    max_depth=2,
    min_samples_leaf=12,
    subsample=0.85,
    colsample=0.8,
    reg_lambda=1.5,
    gamma=0.02,
    random_state=SEED,
)
manual_xgb_clf = ManualXGBClassifier(
    n_estimators=80,
    learning_rate=0.08,
    max_depth=2,
    min_samples_leaf=12,
    subsample=0.85,
    colsample=0.8,
    reg_lambda=1.5,
    gamma=0.02,
    random_state=SEED,
)

manual_xgb_reg.fit(Xr_train, yr_train)
manual_xgb_clf.fit(Xc_train, yc_train)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(manual_xgb_reg.train_scores_, label="Train MSE")
if getattr(manual_xgb_reg, "val_scores_", None):
    axes[0].plot(manual_xgb_reg.val_scores_, label="Validation MSE")
axes[0].set_title(f"Manual XGBoost regressor | best_iteration={manual_xgb_reg.best_iteration_}")
axes[0].set_xlabel("Boosting round")
axes[0].legend()

axes[1].plot(manual_xgb_clf.train_scores_, label="Train log loss")
if getattr(manual_xgb_clf, "val_scores_", None):
    axes[1].plot(manual_xgb_clf.val_scores_, label="Validation log loss")
axes[1].set_title(f"Manual XGBoost classifier | best_iteration={manual_xgb_clf.best_iteration_}")
axes[1].set_xlabel("Boosting round")
axes[1].legend()
plt.tight_layout()

print("Manual XGBoost regression RMSE:", rmse(yr_test, manual_xgb_reg.predict(Xr_test)))
print("Manual XGBoost classification log loss:", log_loss(yc_test, manual_xgb_clf.predict_proba(Xc_test)[:, 1]))


This is the point where the student and the critic should ask the same question:

**Did the extra machinery buy us anything useful?**

The training curves tell us whether the learner is improving in a controlled way. The held-out metrics tell us whether those improvements survive out of sample. Both matter.


## Validating Against the Official Package

A pedagogical implementation is only useful if it agrees qualitatively with a reference implementation. The next cell lines up our manual model with the official `xgboost` package. If the package is not installed in the active environment, the notebook keeps running and reports that the comparison was skipped.

That validation is not about matching every tree exactly. It is about checking whether the simplified learner behaves like the real model in the ways that matter for learning:

- similar prediction patterns,
- similar probability patterns,
- similar held-out performance.


In [ ]:
official_comparison = {}

if HAS_XGBOOST:
    xgb_reg = xgb.XGBRegressor(
        n_estimators=120,
        learning_rate=0.08,
        max_depth=2,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_lambda=1.5,
        gamma=0.02,
        objective="reg:squarederror",
        random_state=SEED,
        eval_metric="rmse",
    )
    xgb_clf = xgb.XGBClassifier(
        n_estimators=120,
        learning_rate=0.08,
        max_depth=2,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_lambda=1.5,
        gamma=0.02,
        objective="binary:logistic",
        random_state=SEED,
        eval_metric="logloss",
    )

    xgb_reg.fit(Xr_train, yr_train)
    xgb_clf.fit(Xc_train, yc_train)

    reg_corr = np.corrcoef(manual_xgb_reg.predict(Xr_test), xgb_reg.predict(Xr_test))[0, 1]
    clf_corr = np.corrcoef(manual_xgb_clf.predict_proba(Xc_test)[:, 1], xgb_clf.predict_proba(Xc_test)[:, 1])[0, 1]

    official_comparison = {
        "regression_prediction_correlation": float(reg_corr),
        "classification_probability_correlation": float(clf_corr),
        "official_regression_rmse": float(rmse(yr_test, xgb_reg.predict(Xr_test))),
        "official_classification_logloss": float(log_loss(yc_test, xgb_clf.predict_proba(Xc_test)[:, 1])),
    }
    pd.Series(official_comparison)
else:
    print(
        "xgboost is not installed in the active environment. "
        "The notebook remains executable, but the official-package comparison is skipped."
    )


This validation step matters because it tells us whether our simplified implementation has learned the *right lesson*. We do not need identical predictions tree-by-tree. We need the manual model to behave like the official package in broad statistical terms.

Student version: "does my simplified picture of XGBoost lead to roughly the same behavior?"

Academic version: "does the reduced implementation preserve the essential optimization logic strongly enough to reproduce the qualitative empirical pattern?"


A second validation question matters too:

**What are we *not* validating here?**

We are not validating the engineering advances that made XGBoost so useful in large-scale practice. Those include:

- exact and approximate split finding strategies,
- weighted quantile sketch for approximate proposals,
- sparsity-aware split finding with learned default directions,
- cache-aware block structures,
- out-of-core training and sharding.

Our notebook validates the statistical learning logic against the official package. It does not try to benchmark or reimplement the full systems design.

That distinction is important. Otherwise students may think they have "implemented XGBoost" when they have really implemented only the core boosting logic.


# Model Comparison

We now compare the whole sequence of models on held-out data.

The point is not to claim one universal winner for every dataset. The point is to see the pattern that motivates the family tree of methods:

- a single tree is flexible but unstable,
- pruning helps,
- bagging reduces variance,
- random forests further reduce correlation,
- gradient boosting and XGBoost-style models often provide the best bias-variance compromise.

The right way to read these tables is historically as well as statistically: each method was invented because the previous one left an identifiable weakness behind.

That is the dual reading we want:

- the student sees the progression,
- the critic checks whether the metrics actually support that progression on these datasets.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def classification_metrics(y_true, proba, pred):
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "LogLoss": log_loss(y_true, proba),
        "ROC_AUC": roc_auc_score(y_true, proba),
        "PR_AUC": average_precision_score(y_true, proba),
        "Brier": brier_score_loss(y_true, proba),
    }


reg_models = {
    "Tree": DecisionTreeRegressor(random_state=SEED),
    "Pruned Tree": DecisionTreeRegressor(random_state=SEED, ccp_alpha=best_reg_alpha),
    "Bagging": bag_reg,
    "Random Forest": rf_reg,
    "Gradient Boosting": gbr,
    "Manual XGBoost": manual_xgb_reg,
}
clf_models = {
    "Tree": DecisionTreeClassifier(random_state=SEED),
    "Pruned Tree": DecisionTreeClassifier(random_state=SEED, ccp_alpha=best_clf_alpha),
    "Bagging": bag_clf,
    "Random Forest": rf_clf,
    "Gradient Boosting": gbc,
    "Manual XGBoost": manual_xgb_clf,
}

if HAS_XGBOOST:
    reg_models["Official XGBoost"] = xgb_reg
    clf_models["Official XGBoost"] = xgb_clf

reg_results = []
for name, model in reg_models.items():
    if name in {"Bagging", "Random Forest", "Gradient Boosting", "Manual XGBoost"}:
        fitted = model
    elif name == "Official XGBoost" and HAS_XGBOOST:
        fitted = model
    else:
        fitted = model.fit(Xr_train, yr_train)
    reg_results.append({"Model": name, **regression_metrics(yr_test, fitted.predict(Xr_test))})

clf_results = []
for name, model in clf_models.items():
    if name in {"Bagging", "Random Forest", "Gradient Boosting", "Manual XGBoost"}:
        fitted = model
    elif name == "Official XGBoost" and HAS_XGBOOST:
        fitted = model
    else:
        fitted = model.fit(Xc_train, yc_train)
    proba = fitted.predict_proba(Xc_test)[:, 1]
    pred = fitted.predict(Xc_test)
    clf_results.append({"Model": name, **classification_metrics(yc_test, proba, pred)})

reg_results_df = pd.DataFrame(reg_results).sort_values("RMSE")
clf_results_df = pd.DataFrame(clf_results).sort_values("LogLoss")

display(reg_results_df.style.format(precision=4))
display(clf_results_df.style.format(precision=4))


When you read the comparison tables, do not focus only on a single "winner." Instead ask:

- how much does pruning help relative to one tree?
- how much variance reduction do we get from bagging and random forests?
- do boosting-style methods improve both fit and probability quality?

Those comparisons teach more than the ranking itself.


# Diagnostics

Better predictive performance does not mean we should stop checking the model.

For regression we care about questions such as:

- do residuals still show structure?
- does variance increase with fitted values?
- do we see extreme tails?

For classification we care about different questions:

- can the model separate the classes?
- are the predicted probabilities calibrated?
- does high accuracy hide poor probability estimates?

So diagnostics should match the learning task rather than being copied mechanically from one setting to another.


In [ ]:
best_reg_model_name = reg_results_df.iloc[0]["Model"]
best_reg_model = reg_models[best_reg_model_name]
reg_pred = best_reg_model.predict(Xr_test)
reg_resid = yr_test - reg_pred

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].scatter(reg_pred, reg_resid, alpha=0.7)
axes[0, 0].axhline(0, color="black", linestyle="--")
axes[0, 0].set_title("Residuals vs fitted")
axes[0, 0].set_xlabel("Fitted values")
axes[0, 0].set_ylabel("Residuals")

sorted_resid = np.sort(reg_resid)
theoretical = np.sort(np.random.normal(0, np.std(reg_resid, ddof=1), size=len(reg_resid)))
axes[0, 1].scatter(theoretical, sorted_resid, alpha=0.7)
axes[0, 1].plot(
    [theoretical.min(), theoretical.max()],
    [theoretical.min(), theoretical.max()],
    color="black",
    linestyle="--",
)
axes[0, 1].set_title("Approximate Q-Q plot")
axes[0, 1].set_xlabel("Theoretical quantiles")
axes[0, 1].set_ylabel("Residual quantiles")

axes[1, 0].hist(reg_resid, bins=20, color="#4c78a8", alpha=0.85)
axes[1, 0].set_title("Residual histogram")
axes[1, 0].set_xlabel("Residual")

axes[1, 1].scatter(reg_pred, np.sqrt(np.abs(reg_resid)), alpha=0.7, color="#f58518")
axes[1, 1].set_title("Scale-location plot")
axes[1, 1].set_xlabel("Fitted values")
axes[1, 1].set_ylabel(r"$\sqrt{|residual|}$")

plt.suptitle(f"Regression diagnostics for: {best_reg_model_name}", y=1.02)
plt.tight_layout()


Residual diagnostics answer a different question from the comparison table.

The table asks: "Which model predicts best on held-out data?"
The residual plots ask: "What kind of mistakes is the best model still making?"

A strong predictive model can still leave visible structure in the residuals, and that is useful scientific information.


In [ ]:
best_clf_model_name = clf_results_df.iloc[0]["Model"]
best_clf_model = clf_models[best_clf_model_name]
clf_proba = best_clf_model.predict_proba(Xc_test)[:, 1]
clf_pred = best_clf_model.predict(Xc_test)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
ConfusionMatrixDisplay.from_predictions(yc_test, clf_pred, ax=axes[0, 0], colorbar=False)
axes[0, 0].set_title("Confusion matrix")

fpr, tpr, _ = roc_curve(yc_test, clf_proba)
axes[0, 1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(yc_test, clf_proba):.3f}")
axes[0, 1].plot([0, 1], [0, 1], linestyle="--", color="black")
axes[0, 1].set_title("ROC curve")
axes[0, 1].set_xlabel("False positive rate")
axes[0, 1].set_ylabel("True positive rate")
axes[0, 1].legend()

precision, recall, _ = precision_recall_curve(yc_test, clf_proba)
axes[1, 0].plot(recall, precision, label=f"AP = {average_precision_score(yc_test, clf_proba):.3f}")
axes[1, 0].set_title("Precision-recall curve")
axes[1, 0].set_xlabel("Recall")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].legend()

CalibrationDisplay.from_predictions(yc_test, clf_proba, n_bins=10, ax=axes[1, 1])
axes[1, 1].set_title("Calibration curve")

plt.suptitle(f"Classification diagnostics for: {best_clf_model_name}", y=1.02)
plt.tight_layout()


For classification, separation and calibration are not the same thing.

- ROC and PR curves tell us whether the model ranks observations well.
- The calibration curve tells us whether probabilities like `0.8` really behave like 80% events.

A careful reader should always keep those two questions separate.


# Discussion and Limitations

Trees and ensembles are popular because they work well on tabular data and can discover nonlinear relationships automatically. But students should keep three limitations in mind.

First, feature importance is not a truth machine. When predictors are correlated, importance rankings can shift depending on the method used.

Second, very strong predictive performance does not guarantee well-calibrated probabilities. A classifier can rank cases correctly while still being overconfident.

Third, our manual XGBoost-style model is useful for understanding the algorithm, not for replacing the official package. Part of XGBoost's practical success comes from engineering, not just from a nicer gain formula.

So the right lesson is not "manual code is enough." The right lesson is "understand the logic first, then trust the industrial implementation for serious work."


# Conclusion

If you are reading this as a student, the main storyline is:

1. one tree is easy to understand but unstable,
2. pruning makes one tree safer,
3. bagging and random forests stabilize trees by averaging,
4. boosting improves a model by adding corrections,
5. XGBoost makes that correction process more explicit and more regularized.

If you are reading this as a critic, the main claim is narrower:

- the notebook explains the statistical logic of modern tree ensembles,
- demonstrates it on both regression and classification,
- and shows that a simplified XGBoost-style learner can behave qualitatively like the official implementation.

That is enough for understanding. It is not the same thing as reproducing the full production system.


# Reproducibility and Commands

The notebook uses fixed random seeds and only built-in scikit-learn datasets. To run the notebook locally:

```bash
jupyter notebook trees_ensembles_xgboost.ipynb
```

To export with the repo helper:

```bash
C:\Users\adamd\conda_envs\quant\python.exe scripts\export_notebook.py --notebook trees_ensembles_xgboost.ipynb --output-dir exports
```

If `xgboost` is not installed, the notebook still runs and the official validation section is skipped. To provision the dependency in a configured environment, use `requirements.txt` or `environment.yml`.


# References

- Breiman, Friedman, Olshen, Stone. *Classification and Regression Trees* (1984). [Google Books](https://books.google.com/books/about/Classification_and_Regression_Trees.html?id=JwQx-WOmSyQC)
- Breiman. *Bagging Predictors* (1996). [DOI](https://doi.org/10.1007/BF00058655)
- Breiman. *Random Forests* (2001). [DOI](https://doi.org/10.1023/A:1010933404324)
- Friedman. *Greedy Function Approximation: A Gradient Boosting Machine* (2001). [DOI](https://doi.org/10.1214/aos/1013203451)
- Chen and Guestrin. *XGBoost: A Scalable Tree Boosting System* (2016). [arXiv PDF](https://arxiv.org/pdf/1603.02754)
- Chen and Guestrin (2016), Sections 2-4 are especially relevant for this notebook: regularized objective, split gain, shrinkage/column subsampling, exact and approximate split finding, sparsity-aware learning, and cache-aware system design.
- Hastie, Tibshirani, Friedman. *The Elements of Statistical Learning* (2nd ed.). [Springer](https://link.springer.com/book/10.1007/978-0-387-84858-7)
- scikit-learn decision tree pruning example. [Documentation](https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html)
- scikit-learn diabetes dataset. [Documentation](https://scikit-learn.org/1.5/modules/generated/sklearn.datasets.load_diabetes.html)
- scikit-learn breast cancer dataset. [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)


# GitHub

This article is also available on [GitHub](https://github.com/adamd1985/quant_research/blob/main/trees_ensembles_xgboost.ipynb)

Kaggle notebook available [here](https://www.kaggle.com/code/addarm/trees-ensembles-and-why-xgboost-works)

# Media

All media used (in the form of code or images) are either solely owned by me, acquired through licensing, or part of the Public Domain and granted use through Creative Commons License.

# CC Licensing and Use

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.
